# TFT Sweep Round 2 — converge the v2-features champion (Kaggle T4, ~2-3 h)
Round 1 (`06_tft_hpo_kaggle.ipynb`) findings:

| trial | features | hidden | val green MAE |
|---|---|---|---|
| v1 (deployed) | v1 | 32 | **1.12** (bar) |
| b_a3_big | v1 | 64 | 1.1239 — statistical tie with v1 |
| **v2_tiebreak** | **v2 (Driver+EngineMaker)** | **64** | **1.083 — beat the bar**, but the cell-8 guard only checked Stage B, so it never deployed (and its ckpt wasn't zipped) |

Round 2 closes that out:
1. **c1** re-trains the v2@h64 champion (same seed 42 -> reproduces ~1.083) so we get its checkpoint.
2. **c2/c3** push the architecture axis (h96/h128 — h64 > h32 both rounds, gradient says try bigger).
3. **c4** sweeps encoder length 12 -> 24 (never tested; stints run ~40 laps, 12 may truncate
   degradation history).
4. Winner (val-green-MAE, val-only selection as always) -> test eval + breakdown/calibration CSVs
   + `models/tft_lap.{ckpt,pt}` + conformal recalibration. Bar stays v1's **1.12**.

Cross-era caveat, stated up front: Driver features degraded TEST in the old v2 run
(driver-as-car-proxy). Selection stays val-based (Errors #22 — never select on test); the winner's
test green is REPORTED honestly either way and goes in the model card next to v1's 1.62.

**Setup identical to notebook 06:** T4 (not P100), Internet ON, Persistence = Files only,
attach the `tft_full_data` parquet dataset, repo pushed to `main`.
Round-1 pace was ~1 min/epoch -> ~2-3 h total. Resumable per-trial via `hpo_round2.csv`.

In [ ]:
# 1. Pinned triangle on top of Kaggle's stock cu128 torch. T4 only (Error 15).
!pip -q install 'pytorch-forecasting==1.7.0' 'lightning==2.6.5' mlflow fastf1 pandera scipy
import torch, pytorch_forecasting as pf, lightning
print('pf', pf.__version__, '| lightning', lightning.__version__, '| torch', torch.__version__)
print('cuda:', torch.cuda.is_available(), '| device:', torch.cuda.get_device_name(0))
assert torch.cuda.get_device_capability(0) in {(7,5),(8,0),(8,6),(9,0)}, \
    'Switch Accelerator to T4 — this GPU arch is not in the torch build.'

In [ ]:
# 2. Clone repo (or pull) + path
import os, sys
REPO = '/kaggle/working/f1-strategist'
if not os.path.exists(REPO):
    !git clone https://github.com/Shreyansh262/f1-strategist.git $REPO
else:
    !cd $REPO && git pull
sys.path.insert(0, REPO)
os.chdir(REPO)
!cd $REPO && git log --oneline -1

In [ ]:
# 3. Copy parquets from the attached dataset into data/raw
import glob, shutil, pathlib
dst = pathlib.Path(REPO) / 'data' / 'raw'
dst.mkdir(parents=True, exist_ok=True)
src_files = glob.glob('/kaggle/input/**/laps_*_r*.parquet', recursive=True)
assert src_files, 'No laps_*_r*.parquet under /kaggle/input/ — attach the data zip as a Dataset first.'
for f in src_files:
    shutil.copy(f, dst / pathlib.Path(f).name)
copied = sorted(p.name for p in dst.glob('laps_*_r*.parquet'))
print(len(copied), 'files | seasons:', sorted({n.split('_')[1] for n in copied}))

In [ ]:
# 4. Data + datasets. ALL round-2 configs use the v2 feature roles (Driver+EngineMaker).
#    Two dataset variants: encoder length 12 (status quo) and 24 (c4). Same min_encoder=3,
#    so the decodable-lap eval set is identical -> val green MAE stays comparable.
import pandas as pd, torch
from pathlib import Path
import src.models.lap_time.train_tft as tt
from src.models.lap_time.train_tft import (
    prepare, green_mae, breakdown, quantile_coverage, export_cpu,
    QUANTILES, STATIC_CATEGORICALS, VAL_SEASONS, TEST_SEASONS)
from src.models.lap_time.train_tft_data import load_tft_data

df = load_tft_data()
BS = 512

def build(enc_len):
    # make_datasets reads the module-level constant at call time
    tt.MAX_ENCODER_LENGTH = enc_len
    tr, va, te = tt.make_datasets(df, static_categoricals=STATIC_CATEGORICALS)
    return dict(
        training=tr,
        tdl=tr.to_dataloader(train=True,  batch_size=BS,   num_workers=4, pin_memory=True),
        vdl=va.to_dataloader(train=False, batch_size=BS*2, num_workers=4),
        test_dl=te.to_dataloader(train=False, batch_size=BS*2, num_workers=4),
    )

DATASETS = {'enc12': build(12), 'enc24': build(24)}
tt.MAX_ENCODER_LENGTH = 12   # restore the default
VAL_RAW  = prepare(df[df['Season'].isin(VAL_SEASONS)])
TEST_RAW = prepare(df[df['Season'].isin(TEST_SEASONS)])
REPORTS  = Path(REPO) / 'reports' / 'lap_time'; REPORTS.mkdir(parents=True, exist_ok=True)
print(f"{len(df)} laps | enc12 train windows {len(DATASETS['enc12']['training'])} "
      f"| enc24 {len(DATASETS['enc24']['training'])}")

In [ ]:
# 5. Trial runner — same contract as round 1, plus per-config dataset choice and the
#    v1-matching EarlyStopping patience 10 (round-1 Stage B used 6; remove that edge).
import gc, time, shutil
import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from pytorch_forecasting import TemporalFusionTransformer
from pytorch_forecasting.metrics import QuantileLoss

HPO_DIR = Path('/kaggle/working/hpo2'); HPO_DIR.mkdir(exist_ok=True)

def run_trial(name, cfg, ds_key, max_epochs=60, patience=10):
    ds = DATASETS[ds_key]
    pl.seed_everything(42, workers=True)
    tft = TemporalFusionTransformer.from_dataset(
        ds['training'],
        learning_rate=cfg['lr'],
        hidden_size=cfg['hidden'],
        attention_head_size=cfg['heads'],
        dropout=cfg['dropout'],
        hidden_continuous_size=cfg['hidden'] // 2,
        loss=QuantileLoss(quantiles=QUANTILES),
        log_interval=-1,
        optimizer='adamw',
        reduce_on_plateau_patience=4,
    )
    n_params = sum(p.numel() for p in tft.parameters())
    ckpt = ModelCheckpoint(dirpath=str(HPO_DIR / name), filename='best',
                           monitor='val_loss', mode='min', save_top_k=1)
    trainer = pl.Trainer(max_epochs=max_epochs,
                         accelerator='gpu' if torch.cuda.is_available() else 'cpu', devices=1,
                         gradient_clip_val=0.1,
                         callbacks=[ckpt, EarlyStopping(monitor='val_loss', patience=patience, mode='min')],
                         logger=False, enable_progress_bar=False, enable_model_summary=False)
    t0 = time.time()
    trainer.fit(tft, ds['tdl'], ds['vdl'])
    best = TemporalFusionTransformer.load_from_checkpoint(ckpt.best_model_path)
    mae = green_mae(best, ds['vdl'], VAL_RAW)   # val only — test waits for the winner
    row = dict(trial=name, **cfg, enc=ds_key, n_params=n_params, epochs=trainer.current_epoch,
               minutes=round((time.time() - t0) / 60, 1),
               val_mae_all=round(mae['mae_all'], 4), val_mae_green=round(mae['mae_green'], 4),
               ckpt=ckpt.best_model_path)
    del tft, best, trainer; gc.collect(); torch.cuda.empty_cache()
    return row

def run_stage(configs, csv_path):
    done = pd.read_csv(csv_path) if csv_path.exists() else pd.DataFrame()
    for name, cfg, ds_key in configs:
        if not done.empty and name in set(done['trial']):
            print('skip (already done):', name); continue
        print(f'>>> {name} {cfg} [{ds_key}]', flush=True)
        row = run_trial(name, cfg, ds_key)
        done = pd.concat([done, pd.DataFrame([row])], ignore_index=True)
        done.to_csv(csv_path, index=False)
        print(row, flush=True)
    return done

In [ ]:
# 6. Round-2 configs — all v2 features, all converged (max 60 epochs, patience 10).
ROUND2 = [
    ('c1_v2_h64',       dict(hidden=64,  lr=1e-3, dropout=0.20, heads=4), 'enc12'),  # reproduce champion
    ('c2_v2_h96',       dict(hidden=96,  lr=1e-3, dropout=0.20, heads=4), 'enc12'),
    ('c3_v2_h128',      dict(hidden=128, lr=1e-3, dropout=0.25, heads=4), 'enc12'),  # more dropout at size
    ('c4_v2_h64_enc24', dict(hidden=64,  lr=1e-3, dropout=0.20, heads=4), 'enc24'),
]
CSV_R2 = REPORTS / 'hpo_round2.csv'
round2 = run_stage(ROUND2, CSV_R2)
print(round2.sort_values('val_mae_green').to_string(index=False))

In [ ]:
# 7. Winner: val-green selection across round 2; deploy guard = v1 bar 1.12.
#    Test set touched here only, for the winner — reported honestly win or lose vs v1's 1.62.
V1_VAL_GREEN, V1_TEST_GREEN = 1.12, 1.62
round2 = pd.read_csv(CSV_R2).sort_values('val_mae_green')
win = round2.iloc[0]
print(round2.to_string(index=False), '\n')

if win.val_mae_green < V1_VAL_GREEN:
    print(f'WINNER {win.trial}: val green {win.val_mae_green:.3f} beats v1 {V1_VAL_GREEN} -> deploying')
    best = TemporalFusionTransformer.load_from_checkpoint(win.ckpt)
    ds = DATASETS[win.enc]
    bd_frames, cov_rows = [], []
    for split, dl, raw in [('val', ds['vdl'], VAL_RAW), ('test', ds['test_dl'], TEST_RAW)]:
        bd = breakdown(best, dl, raw); bd.insert(0, 'split', split); bd_frames.append(bd)
        cov_rows.append({'split': split, **quantile_coverage(best, dl, raw)})
    bd_all = pd.concat(bd_frames, ignore_index=True)
    bd_all.to_csv(REPORTS / 'tft_breakdown.csv', index=False)
    pd.DataFrame(cov_rows).to_csv(REPORTS / 'tft_calibration.csv', index=False)
    tg = bd_all[(bd_all.split == 'test') & (bd_all.scope == 'overall')]['mae_green'].iloc[0]
    verdict = 'IMPROVES on' if tg < V1_TEST_GREEN else 'REGRESSES vs'
    print(f'Cross-era check: winner test green {tg:.3f} {verdict} v1 {V1_TEST_GREEN} '
          '(reported, not used for selection — Errors #22)')
    MODELS = Path(REPO) / 'models'; MODELS.mkdir(exist_ok=True)
    shutil.copy(win.ckpt, MODELS / 'tft_lap.ckpt')
    export_cpu(best, MODELS / 'tft_lap.pt')
    from src.models.lap_time.recalibrate import main as recal
    recal()
else:
    print(f'No round-2 config beat v1 ({V1_VAL_GREEN}) on val green — KEEP v1, do not overwrite models/.')

In [ ]:
# 8. Bundle artifacts (includes the deployed winner ckpt via models/).
import os
for fn in ('hpo_round2.csv', 'tft_breakdown.csv', 'tft_calibration.csv', 'tft_recalibration.csv'):
    p = REPORTS / fn
    if p.exists():
        print('\n===', fn, '===')
        print(pd.read_csv(p).to_string(index=False))
(Path(REPO) / 'models').mkdir(exist_ok=True)
!cd $REPO && zip -qr /kaggle/working/tft_hpo2_artifacts.zip models reports/lap_time
print('\nDownload: /kaggle/working/tft_hpo2_artifacts.zip')

## After downloading (LOCAL steps)
**If a winner deployed (expected — c1 alone should reproduce ~1.083):**
1. Unzip `tft_hpo2_artifacts.zip` into the repo root — BUT first zip the current local v1
   artifacts (`models/tft_lap.*` + `models/tft_calibration.json`) as `tft_v1_artifacts_backup.zip`
   (lesson from the v2 episode: keep the previous champion's zip).
2. `python -m src.models.lap_time.evaluate` -> refreshed `model_comparison.csv`.
3. `pytest -q` — all green.
4. Model card + MASTER_CONTEXT: new hparams, val AND test green (test regression, if any, documented
   as the driver-as-proxy cost — val-based selection, both numbers shown).

**If no winner:** unzip only `reports/lap_time/hpo_round2.csv`; v1 confirmed across two sweep rounds.